# Journal Extension J2 — DINOv2 on the Original 80/10/10 Thesis Split

## Goal

Train a strong modern SSL baseline, **DINOv2 ViT-S/14**, on the **exact J1-locked thesis split** that produced the proposed model's ~83.95% internal BACC.

This makes the comparison apples-to-apples.

### Protocol
- TRAIN: exact J1 locked TRAIN set
- Validation: exact J1 locked validation set
- Test: exact J1 locked 2,534-image test set
- Checkpoint selection: **validation BACC only**
- Test set: reporting only
- Seeds: **21, 42, 62**
- Multi-seed DINO result: mean probability across the three validation-selected checkpoints
- DINO is a comparison baseline; it is **not automatically added to the proposed ensemble**

## Kaggle inputs required

### 1. J1 saved output — REQUIRED
Attach the saved output of:

`J1_Original_Protocol_Lock_Reproducibility.ipynb`

It must contain either:
- `J1_PROTOCOL_LOCK_COMPLETE.json`, and
- `j1_original_protocol_lock/tables/original_80_10_10_locked_ids.csv`

The J1 package also contains the frozen proposed-model probabilities, which this notebook uses for direct comparison.

### 2. ISIC 2019 training images — REQUIRED
Attach the **same ISIC 2019 training image dataset** used by the thesis.

The notebook matches images by the `ISIC_xxxxxxx` filename stem, so the directory name can differ.

### 3. Internet — REQUIRED for first DINOv2 download
Turn Kaggle Internet **ON** so `torch.hub` can obtain the official DINOv2 ViT-S/14 weights.

## Not required
You do not need to attach:
- confidence-gated ensemble separately (J1 already copied the needed artifacts),
- PAD-UFES-20,
- official ISIC test set,
- grouped Stage-1/2/3 outputs,
- previous grouped-split DINO output.

## Outputs
- per-seed validation/test metrics
- 3-seed mean DINO probabilities
- representative validation-selected DINO embeddings
- classification report
- confusion matrix
- comparison with the frozen 83.95% proposed model
- selection audit
- `J2_DINO_ORIGINAL_SPLIT_COMPLETE.json`
- `J2_DINO_ORIGINAL_SPLIT_RESULTS.zip`

In [1]:
# ============================================================
# CELL 0 — ENVIRONMENT PREFLIGHT
# No torchvision is used anywhere in this notebook.
# ============================================================
import os
import sys
import subprocess

PYTHON = sys.executable

def run_check(script):
    r = subprocess.run(
        [PYTHON, "-c", script],
        text=True,
        capture_output=True,
    )
    if r.stdout:
        print(r.stdout)
    if r.stderr:
        print(r.stderr)
    return r.returncode == 0

print("=" * 96)
print("J2 ENVIRONMENT PREFLIGHT")
print("=" * 96)

PIL_TEST = r'''
from PIL import Image, ImageEnhance
import PIL
print("Pillow:", PIL.__version__)
'''

if not run_check(PIL_TEST):
    print("Repairing Pillow...")
    subprocess.run(
        [
            PYTHON, "-m", "pip", "install",
            "--no-cache-dir", "--upgrade", "--force-reinstall",
            "Pillow==11.3.0",
        ],
        check=True,
    )
    if not run_check(PIL_TEST):
        raise RuntimeError("Pillow remains unusable.")

CUDA_TEST = r'''
import torch
print("torch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA unavailable.")
print("GPU:", torch.cuda.get_device_name(0))
print("Capability:", torch.cuda.get_device_capability(0))
x = torch.randn(2, 3, 32, 32, device="cuda")
m = torch.nn.Conv2d(3, 8, 3, padding=1).cuda()
y = m(x)
torch.cuda.synchronize()
print("CUDA Conv2d: PASSED", tuple(y.shape))
'''

cuda_ok = run_check(CUDA_TEST)

if not cuda_ok:
    print("\nCurrent PyTorch build cannot execute kernels on this GPU.")
    print("Installing the same conservative CUDA-compatible build used by the corrected DINO notebook...")

    subprocess.run(
        [
            PYTHON, "-m", "pip", "uninstall", "-y",
            "torchvision", "torchaudio", "xformers",
        ],
        check=False,
    )

    subprocess.run(
        [
            PYTHON, "-m", "pip", "install",
            "--no-cache-dir", "--upgrade", "--force-reinstall",
            "torch==2.7.1",
            "--index-url", "https://download.pytorch.org/whl/cu118",
        ],
        check=True,
    )

    cuda_ok = run_check(CUDA_TEST)

if not cuda_ok:
    raise RuntimeError(
        "CUDA still cannot execute PyTorch kernels. "
        "Use a fresh Kaggle GPU session (T4/L4/A100 if available) and rerun."
    )

subprocess.run(
    [PYTHON, "-m", "pip", "uninstall", "-y", "xformers"],
    check=False,
)

print("Environment preflight: PASSED")

J2 ENVIRONMENT PREFLIGHT
Pillow: 11.3.0

torch: 2.10.0+cu128
CUDA runtime: 12.8
CUDA available: True
GPU: Tesla P100-PCIE-16GB
Capability: (6, 0)

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GP

Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 383.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 375.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 179.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 256.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 310.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 312.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 306.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 340.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 362.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 MB 369.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 147.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 905.2/9

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
easyocr 1.7.2 requires torchvision>=0.5, which is not installed.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
timm 1.0.26 requires torchvision, which is not installed.
fastai 2.8.7 requires torchvision>=0.11, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.4.0 which is incompatible.


torch: 2.7.1+cu118
CUDA runtime: 11.8
CUDA available: True
GPU: Tesla P100-PCIE-16GB
Capability: (6, 0)
CUDA Conv2d: PASSED (2, 8, 32, 32)

Environment preflight: PASSED


In [2]:
# ============================================================
# CELL 1 — IMPORTS / CONFIG
# ============================================================
import gc
import json
import math
import time
import random
import hashlib
import shutil
import zipfile
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageEnhance
import PIL

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
)
from sklearn.preprocessing import label_binarize

KAGGLE_INPUT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")

OUTPUT_DIR = WORKING / "j2_dinov2_original_split"
MODEL_DIR = OUTPUT_DIR / "models"
TABLE_DIR = OUTPUT_DIR / "tables"
FIG_DIR = OUTPUT_DIR / "figures"
FEATURE_DIR = OUTPUT_DIR / "features"
AUDIT_DIR = OUTPUT_DIR / "audit"

for d in [OUTPUT_DIR, MODEL_DIR, TABLE_DIR, FIG_DIR, FEATURE_DIR, AUDIT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

NUM_CLASSES = 8
CLASS_NAMES = ["MEL","NV","BCC","AK","BKL","DF","VASC","SCC"]
IMG_SIZE = 224

SEEDS = [21, 42, 62]
BATCH_SIZE = 16
NUM_WORKERS = 2

LINEAR_WARMUP_EPOCHS = 3
FINETUNE_EPOCHS = 10
PATIENCE = 3

DINO_MODEL_NAME = "dinov2_vits14"

if not torch.cuda.is_available():
    raise RuntimeError("CUDA unavailable after preflight.")

DEVICE = torch.device("cuda")
AMP_ENABLED = True
torch.backends.cudnn.benchmark = True

# In-kernel CUDA smoke test
_x = torch.randn(1, 3, 32, 32, device=DEVICE)
_m = nn.Conv2d(3, 4, 3, padding=1).to(DEVICE)
_y = _m(_x)
torch.cuda.synchronize()
del _x, _m, _y
torch.cuda.empty_cache()

try:
    CUDA_ARCH_FLAGS = torch._C._cuda_getArchFlags()
except Exception:
    CUDA_ARCH_FLAGS = "unavailable"

print("=" * 96)
print("J2 — DINOv2 / ORIGINAL 80-10-10 SPLIT")
print("=" * 96)
print("PyTorch    :", torch.__version__)
print("Torch CUDA :", torch.version.cuda)
print("Pillow     :", PIL.__version__)
print("GPU        :", torch.cuda.get_device_name(0))
print("Capability :", torch.cuda.get_device_capability(0))
print("Arch flags :", CUDA_ARCH_FLAGS)
print("Seeds      :", SEEDS)
print("Batch size :", BATCH_SIZE)

J2 — DINOv2 / ORIGINAL 80-10-10 SPLIT
PyTorch    : 2.7.1+cu118
Torch CUDA : 11.8
Pillow     : 11.3.0
GPU        : Tesla P100-PCIE-16GB
Capability : (6, 0)
Arch flags : sm_50 sm_60 sm_70 sm_75 sm_80 sm_86 sm_37 sm_90
Seeds      : [21, 42, 62]
Batch size : 16


In [3]:
# ============================================================
# CELL 2 — FIND J1 LOCK PACKAGE
# ============================================================

def find_first(filename):
    hits = sorted(
        KAGGLE_INPUT.rglob(filename),
        key=lambda p: (len(p.parts), str(p))
    )
    return hits[0] if hits else None

J1_MARKER = find_first("J1_PROTOCOL_LOCK_COMPLETE.json")

if J1_MARKER is None:
    raise FileNotFoundError(
        "J1_PROTOCOL_LOCK_COMPLETE.json not found. "
        "Attach the saved J1 notebook output."
    )

J1_ROOT = J1_MARKER.parent

LOCKED_IDS_CANDIDATES = sorted(
    KAGGLE_INPUT.rglob("original_80_10_10_locked_ids.csv"),
    key=lambda p: (len(p.parts), str(p))
)

if not LOCKED_IDS_CANDIDATES:
    raise FileNotFoundError(
        "original_80_10_10_locked_ids.csv not found in the J1 input."
    )

LOCKED_IDS_CSV = LOCKED_IDS_CANDIDATES[0]

# Prefer J1-copied frozen thesis probabilities.
FINAL_PROB_CANDIDATES = sorted(
    [
        p for p in KAGGLE_INPUT.rglob("final_internal_probabilities.npz")
        if "j1_original_protocol_lock" in str(p)
        or "source_artifacts" in str(p)
    ],
    key=lambda p: (len(p.parts), str(p))
)

if not FINAL_PROB_CANDIDATES:
    FINAL_PROB_CANDIDATES = sorted(
        KAGGLE_INPUT.rglob("final_internal_probabilities.npz"),
        key=lambda p: (len(p.parts), str(p))
    )

if not FINAL_PROB_CANDIDATES:
    raise FileNotFoundError(
        "Frozen final_internal_probabilities.npz not found. "
        "The J1 saved output should contain it."
    )

FINAL_PROB_NPZ = FINAL_PROB_CANDIDATES[0]

COMPARISON_CANDIDATES = sorted(
    [
        p for p in KAGGLE_INPUT.rglob("internal_test_all_models_comparison.csv")
        if "j1_original_protocol_lock" in str(p)
        or "source_artifacts" in str(p)
    ],
    key=lambda p: (len(p.parts), str(p))
)

COMPARISON_CSV = COMPARISON_CANDIDATES[0] if COMPARISON_CANDIDATES else None

with open(J1_MARKER, "r") as f:
    J1_INFO = json.load(f)

LOCKED_DF = pd.read_csv(LOCKED_IDS_CSV)

expected_counts = {
    "TRAIN": 20264,
    "VAL": 2533,
    "TEST": 2534,
}

actual_counts = LOCKED_DF["split"].value_counts().to_dict()

for split_name, expected in expected_counts.items():
    if int(actual_counts.get(split_name, -1)) != expected:
        raise RuntimeError(
            f"J1 split size mismatch for {split_name}: "
            f"{actual_counts.get(split_name)} != {expected}"
        )

print("J1 marker     :", J1_MARKER)
print("Locked IDs    :", LOCKED_IDS_CSV)
print("Frozen probs  :", FINAL_PROB_NPZ)
print("Protocol hash :", J1_INFO.get("combined_protocol_sha256"))
print("Split counts  :", actual_counts)

J1 marker     : /kaggle/input/notebooks/anirbanroysudipto/journal-extension-j1/J1_PROTOCOL_LOCK_COMPLETE.json
Locked IDs    : /kaggle/input/notebooks/anirbanroysudipto/journal-extension-j1/j1_original_protocol_lock/tables/original_80_10_10_locked_ids.csv
Frozen probs  : /kaggle/input/notebooks/anirbanroysudipto/journal-extension-j1/j1_original_protocol_lock/source_artifacts/final_internal_probabilities.npz
Protocol hash : b483ced043583491ed0793a5e80ecd1192055230393b6603200a3441d04f5766
Split counts  : {'TRAIN': 20264, 'TEST': 2534, 'VAL': 2533}


In [4]:
# ============================================================
# CELL 3 — DISCOVER THE ISIC 2019 TRAINING IMAGES BY ID
# ============================================================

needed_ids = set(LOCKED_DF["image_id"].astype(str))
image_map = {}
duplicate_candidates = {}

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("Scanning /kaggle/input for the locked ISIC image IDs...")

for p in KAGGLE_INPUT.rglob("*"):
    if not p.is_file():
        continue
    if p.suffix.lower() not in IMAGE_EXTS:
        continue

    stem = p.stem

    if stem not in needed_ids:
        continue

    if stem not in image_map:
        image_map[stem] = str(p)
    else:
        duplicate_candidates.setdefault(stem, [image_map[stem]])
        duplicate_candidates[stem].append(str(p))

missing_ids = sorted(needed_ids - set(image_map))

print("Locked IDs       :", len(needed_ids))
print("Matched images   :", len(image_map))
print("Missing images   :", len(missing_ids))
print("Duplicate stems  :", len(duplicate_candidates))

if missing_ids:
    raise FileNotFoundError(
        f"{len(missing_ids)} locked ISIC images were not found. "
        f"Examples: {missing_ids[:20]}. "
        "Attach the ISIC 2019 training image dataset used by the thesis."
    )

LOCKED_DF["filepath"] = LOCKED_DF["image_id"].astype(str).map(image_map)

if LOCKED_DF["filepath"].isna().any():
    raise RuntimeError("Image mapping failed unexpectedly.")

TRAIN_DF = LOCKED_DF.loc[
    LOCKED_DF["split"] == "TRAIN"
].reset_index(drop=True)

VAL_DF = LOCKED_DF.loc[
    LOCKED_DF["split"] == "VAL"
].reset_index(drop=True)

TEST_DF = LOCKED_DF.loc[
    LOCKED_DF["split"] == "TEST"
].reset_index(drop=True)

pd.DataFrame(
    [{"image_id": k, "candidate_paths": " || ".join(v)}
     for k, v in duplicate_candidates.items()]
).to_csv(
    AUDIT_DIR / "duplicate_image_stem_candidates.csv",
    index=False
)

print("Image discovery: PASSED")
print("TRAIN / VAL / TEST:", len(TRAIN_DF), len(VAL_DF), len(TEST_DF))

Scanning /kaggle/input for the locked ISIC image IDs...
Locked IDs       : 25331
Matched images   : 25331
Missing images   : 0
Duplicate stems  : 1
Image discovery: PASSED
TRAIN / VAL / TEST: 20264 2533 2534


In [5]:
# ============================================================
# CELL 4 — CLASS WEIGHTS
# ============================================================

counts = (
    TRAIN_DF["label"]
    .value_counts()
    .sort_index()
    .reindex(range(NUM_CLASSES))
)

if counts.isna().any() or (counts <= 0).any():
    raise RuntimeError("One or more classes are missing from TRAIN.")

# Balanced class weights, normalized so mean weight = 1.
raw_weights = len(TRAIN_DF) / (NUM_CLASSES * counts.to_numpy(dtype=np.float64))
raw_weights = raw_weights / np.mean(raw_weights)

CLASS_WEIGHT_VECTOR = raw_weights.astype(np.float32)

CLASS_WEIGHTS = torch.tensor(
    CLASS_WEIGHT_VECTOR,
    dtype=torch.float32,
    device=DEVICE,
)

CLASS_WEIGHT_DF = pd.DataFrame({
    "class_id": np.arange(NUM_CLASSES),
    "class_name": CLASS_NAMES,
    "train_count": counts.to_numpy(),
    "class_weight": CLASS_WEIGHT_VECTOR,
})

display(CLASS_WEIGHT_DF)

CLASS_WEIGHT_DF.to_csv(
    TABLE_DIR / "dinov2_training_class_weights.csv",
    index=False
)

   class_id class_name  train_count  class_weight
0         0        MEL         3618      0.148948
1         1         NV        10300      0.052320
2         2        BCC         2658      0.202745
3         3         AK          694      0.776507
4         4        BKL         2099      0.256739
5         5         DF          191      2.821443
6         6       VASC          202      2.667800
7         7        SCC          502      1.073497

In [6]:
# ============================================================
# CELL 5 — CUSTOM IMAGE TRANSFORMS
# No torchvision dependency.
# ============================================================

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def resize_bicubic(img, size=IMG_SIZE):
    return img.resize(
        (size, size),
        resample=Image.Resampling.BICUBIC
    )

def random_resized_crop(
    img,
    output_size=IMG_SIZE,
    scale=(0.80, 1.00),
    ratio=(3/4, 4/3),
):
    width, height = img.size
    area = width * height
    log_ratio = (math.log(ratio[0]), math.log(ratio[1]))

    for _ in range(10):
        target_area = area * random.uniform(*scale)
        aspect = math.exp(random.uniform(*log_ratio))
        crop_w = int(round(math.sqrt(target_area * aspect)))
        crop_h = int(round(math.sqrt(target_area / aspect)))

        if 0 < crop_w <= width and 0 < crop_h <= height:
            left = random.randint(0, width - crop_w)
            top = random.randint(0, height - crop_h)
            cropped = img.crop(
                (left, top, left + crop_w, top + crop_h)
            )
            return resize_bicubic(cropped, output_size)

    side = min(width, height)
    left = (width - side) // 2
    top = (height - side) // 2
    cropped = img.crop((left, top, left + side, top + side))
    return resize_bicubic(cropped, output_size)

def jitter_hue(img, max_delta=0.02):
    hsv = np.array(img.convert("HSV"), dtype=np.uint8)
    shift = int(round(random.uniform(-max_delta, max_delta) * 255))
    h = hsv[..., 0].astype(np.int16)
    hsv[..., 0] = ((h + shift) % 256).astype(np.uint8)
    return Image.fromarray(hsv, mode="HSV").convert("RGB")

def color_jitter(img):
    brightness = random.uniform(0.85, 1.15)
    contrast = random.uniform(0.85, 1.15)
    saturation = random.uniform(0.90, 1.10)

    ops = [
        lambda x: ImageEnhance.Brightness(x).enhance(brightness),
        lambda x: ImageEnhance.Contrast(x).enhance(contrast),
        lambda x: ImageEnhance.Color(x).enhance(saturation),
    ]
    random.shuffle(ops)

    for op in ops:
        img = op(img)

    return jitter_hue(img, 0.02)

def pil_to_normalized_tensor(img):
    arr = np.asarray(img, dtype=np.float32) / 255.0

    if arr.ndim != 3 or arr.shape[2] != 3:
        raise ValueError(f"Expected RGB image, got {arr.shape}")

    arr = (arr - IMAGENET_MEAN) / IMAGENET_STD
    arr = np.transpose(arr, (2, 0, 1))
    arr = np.ascontiguousarray(arr).copy()

    return torch.from_numpy(arr).float()

class TrainTransform:
    def __call__(self, img):
        img = random_resized_crop(img)

        if random.random() < 0.5:
            img = img.transpose(Image.Transpose.FLIP_LEFT_RIGHT)

        if random.random() < 0.5:
            img = img.transpose(Image.Transpose.FLIP_TOP_BOTTOM)

        img = color_jitter(img)
        return pil_to_normalized_tensor(img)

class EvalTransform:
    def __call__(self, img):
        return pil_to_normalized_tensor(
            resize_bicubic(img)
        )

TRAIN_TRANSFORM = TrainTransform()
EVAL_TRANSFORM = EvalTransform()

_test_img = Image.fromarray(
    np.zeros((224, 224, 3), dtype=np.uint8)
)
_test_tensor = TRAIN_TRANSFORM(_test_img)
assert tuple(_test_tensor.shape) == (3, 224, 224)

print("Transform smoke test: PASSED", tuple(_test_tensor.shape))

Transform smoke test: PASSED (3, 224, 224)


/tmp/ipykernel_23/258848220.py:50: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return Image.fromarray(hsv, mode="HSV").convert("RGB")


In [7]:
# ============================================================
# CELL 6 — DATASET / LOADERS
# ============================================================

class SkinDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        with Image.open(row["filepath"]) as im:
            image = im.convert("RGB")

        image = self.transform(image)

        return image, int(row["label"]), str(row["image_id"])

def make_loader(df, training, seed):
    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        SkinDataset(
            df,
            TRAIN_TRANSFORM if training else EVAL_TRANSFORM
        ),
        batch_size=BATCH_SIZE,
        shuffle=training,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
        drop_last=False,
        generator=generator if training else None,
    )

VAL_LOADER = make_loader(VAL_DF, False, 0)
TEST_LOADER = make_loader(TEST_DF, False, 0)

xb, yb, ids = next(iter(VAL_LOADER))
print("Loader smoke test:", tuple(xb.shape), tuple(yb.shape), len(ids))

Loader smoke test: (16, 3, 224, 224) (16,) 16


In [8]:
# ============================================================
# CELL 7 — LOAD OFFICIAL PRETRAINED DINOv2 ViT-S/14
# ============================================================

try:
    pretrained_backbone = torch.hub.load(
        "facebookresearch/dinov2",
        DINO_MODEL_NAME,
        pretrained=True,
        trust_repo=True,
    )
except Exception as e:
    raise RuntimeError(
        "Could not load official DINOv2. "
        "Turn Kaggle Internet ON for the first run. "
        f"Original error: {type(e).__name__}: {e}"
    )

EMBED_DIM = int(getattr(pretrained_backbone, "embed_dim", 384))

# Keep one clean pretrained state on CPU.
PRETRAINED_STATE = {
    k: v.detach().cpu().clone()
    for k, v in pretrained_backbone.state_dict().items()
}

del pretrained_backbone
gc.collect()
torch.cuda.empty_cache()

class DINOClassifier(nn.Module):
    def __init__(self, backbone, embed_dim, num_classes):
        super().__init__()
        self.backbone = backbone
        self.dropout = nn.Dropout(0.30)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x, return_features=False):
        features = self.backbone(x)
        logits = self.head(self.dropout(features))

        if return_features:
            return logits, features

        return logits

def build_fresh_model():
    backbone = torch.hub.load(
        "facebookresearch/dinov2",
        DINO_MODEL_NAME,
        pretrained=False,
        trust_repo=True,
    )
    backbone.load_state_dict(PRETRAINED_STATE, strict=True)

    model = DINOClassifier(
        backbone,
        EMBED_DIM,
        NUM_CLASSES,
    ).to(DEVICE)

    return model

# DINO CUDA smoke test
_smoke_model = build_fresh_model()
_smoke_model.eval()

with torch.no_grad():
    dummy = torch.randn(
        1, 3, IMG_SIZE, IMG_SIZE,
        device=DEVICE
    )
    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=True,
    ):
        logits = _smoke_model(dummy)
    torch.cuda.synchronize()

assert tuple(logits.shape) == (1, NUM_CLASSES)

TOTAL_PARAMS = sum(p.numel() for p in _smoke_model.parameters())
BACKBONE_PARAMS = sum(
    p.numel() for p in _smoke_model.backbone.parameters()
)

print("DINO CUDA smoke test: PASSED")
print("Embedding dim:", EMBED_DIM)
print("Total params :", f"{TOTAL_PARAMS:,}")

del _smoke_model, dummy, logits
gc.collect()
torch.cuda.empty_cache()

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 371MB/s]
Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


DINO CUDA smoke test: PASSED
Embedding dim: 384
Total params : 22,059,656


In [9]:
# ============================================================
# CELL 8 — METRICS / PREDICTION HELPERS
# ============================================================

LOSS_FN = nn.CrossEntropyLoss(
    weight=CLASS_WEIGHTS
)

def normalize_prob(p):
    p = np.asarray(p, dtype=np.float64)
    p = np.clip(p, 1e-12, None)
    p /= p.sum(axis=1, keepdims=True)
    return p.astype(np.float32)

def metrics_dict(y, prob):
    prob = normalize_prob(prob)
    pred = np.argmax(prob, axis=1)

    y_bin = label_binarize(
        y,
        classes=np.arange(NUM_CLASSES)
    )

    try:
        auc = float(
            roc_auc_score(
                y_bin,
                prob,
                average="macro",
                multi_class="ovr",
            )
        )
    except Exception:
        auc = np.nan

    return {
        "accuracy": float(accuracy_score(y, pred)),
        "bacc": float(balanced_accuracy_score(y, pred)),
        "macro_precision": float(
            precision_score(y, pred, average="macro", zero_division=0)
        ),
        "macro_recall": float(
            recall_score(y, pred, average="macro", zero_division=0)
        ),
        "macro_f1": float(
            f1_score(y, pred, average="macro", zero_division=0)
        ),
        "weighted_f1": float(
            f1_score(y, pred, average="weighted", zero_division=0)
        ),
        "auc": auc,
    }

@torch.no_grad()
def predict_loader(model, loader, return_features=False):
    model.eval()

    ys = []
    probs = []
    ids = []
    feats = []

    for images, labels, image_ids in loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=AMP_ENABLED,
        ):
            if return_features:
                logits, features = model(
                    images,
                    return_features=True
                )
                feats.append(
                    features.float().cpu().numpy()
                )
            else:
                logits = model(images)

        prob = torch.softmax(
            logits.float(),
            dim=1
        )

        ys.append(labels.numpy())
        probs.append(prob.cpu().numpy())
        ids.extend(list(image_ids))

    y = np.concatenate(ys).astype(np.int32)
    p = normalize_prob(np.concatenate(probs))
    ids = np.asarray(ids)

    if return_features:
        features = np.concatenate(feats).astype(np.float32)
        return y, p, ids, features

    return y, p, ids

In [10]:
# ============================================================
# CELL 9 — TRAINING HELPERS
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def make_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)

def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    losses = []

    for images, labels, _ in loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )
        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=AMP_ENABLED,
        ):
            logits = model(images)
            loss = LOSS_FN(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        losses.append(float(loss.detach().cpu()))

    return float(np.mean(losses))

def fit_phase(
    model,
    train_loader,
    optimizer,
    epochs,
    checkpoint_path,
    seed,
    phase_name,
):
    scaler = make_scaler()

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=max(1, epochs),
    )

    best_bacc = -np.inf
    best_f1 = -np.inf
    stale = 0
    rows = []

    for epoch in range(1, epochs + 1):
        t0 = time.time()

        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            scaler,
        )

        yv, pv, _ = predict_loader(
            model,
            VAL_LOADER
        )

        m = metrics_dict(yv, pv)

        row = {
            "seed": seed,
            "phase": phase_name,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_accuracy": m["accuracy"],
            "val_bacc": m["bacc"],
            "val_macro_f1": m["macro_f1"],
            "seconds": time.time() - t0,
        }
        rows.append(row)

        print(
            f"seed={seed} | {phase_name} | "
            f"epoch {epoch:02d}/{epochs} | "
            f"loss={train_loss:.4f} | "
            f"VAL BACC={m['bacc']:.4f} | "
            f"F1={m['macro_f1']:.4f}"
        )

        improved = (
            m["bacc"] > best_bacc + 1e-6
            or (
                abs(m["bacc"] - best_bacc) <= 1e-6
                and m["macro_f1"] > best_f1 + 1e-6
            )
        )

        if improved:
            best_bacc = m["bacc"]
            best_f1 = m["macro_f1"]
            stale = 0

            torch.save(
                {
                    "seed": seed,
                    "phase": phase_name,
                    "epoch": epoch,
                    "val_bacc": best_bacc,
                    "val_macro_f1": best_f1,
                    "state_dict": model.state_dict(),
                },
                checkpoint_path,
            )
        else:
            stale += 1

        scheduler.step()

        if stale >= PATIENCE:
            print("Early stopping:", phase_name, "seed", seed)
            break

    ckpt = torch.load(
        checkpoint_path,
        map_location=DEVICE
    )

    model.load_state_dict(
        ckpt["state_dict"]
    )

    return pd.DataFrame(rows), ckpt

In [11]:
# ============================================================
# CELL 10 — 3-SEED DINOv2 TRAINING
# ============================================================

seed_summaries = []
all_histories = []

val_probs_by_seed = {}
test_probs_by_seed = {}
val_features_by_seed = {}
test_features_by_seed = {}
val_ids_by_seed = {}
test_ids_by_seed = {}

for seed in SEEDS:
    print("\n" + "=" * 96)
    print("DINOv2 SEED", seed)
    print("=" * 96)

    set_seed(seed)

    train_loader = make_loader(
        TRAIN_DF,
        True,
        seed
    )

    model = build_fresh_model()

    # -------------------------
    # Phase A: head-only warmup
    # -------------------------
    for p in model.backbone.parameters():
        p.requires_grad = False

    for p in model.head.parameters():
        p.requires_grad = True

    warmup_ckpt = (
        MODEL_DIR /
        f"seed_{seed}_linear_warmup_best.pt"
    )

    warmup_optimizer = torch.optim.AdamW(
        model.head.parameters(),
        lr=1e-3,
        weight_decay=1e-4,
    )

    warmup_history, warmup_best = fit_phase(
        model,
        train_loader,
        warmup_optimizer,
        LINEAR_WARMUP_EPOCHS,
        warmup_ckpt,
        seed,
        "linear_warmup",
    )

    all_histories.append(warmup_history)

    # -------------------------
    # Phase B: full fine-tuning
    # -------------------------
    for p in model.parameters():
        p.requires_grad = True

    finetune_ckpt = (
        MODEL_DIR /
        f"seed_{seed}_finetuned_best.pt"
    )

    optimizer = torch.optim.AdamW(
        [
            {
                "params": model.backbone.parameters(),
                "lr": 1e-5,
            },
            {
                "params": model.head.parameters(),
                "lr": 1e-4,
            },
        ],
        weight_decay=0.05,
    )

    finetune_history, finetune_best = fit_phase(
        model,
        train_loader,
        optimizer,
        FINETUNE_EPOCHS,
        finetune_ckpt,
        seed,
        "full_finetune",
    )

    all_histories.append(finetune_history)

    # Frozen evaluation from the validation-selected checkpoint.
    yv, pv, ids_v, fv = predict_loader(
        model,
        VAL_LOADER,
        return_features=True,
    )

    yt, pt, ids_t, ft = predict_loader(
        model,
        TEST_LOADER,
        return_features=True,
    )

    locked_val_y = VAL_DF["label"].to_numpy(np.int32)
    locked_test_y = TEST_DF["label"].to_numpy(np.int32)
    locked_val_ids = VAL_DF["image_id"].astype(str).to_numpy()
    locked_test_ids = TEST_DF["image_id"].astype(str).to_numpy()

    if not np.array_equal(yv, locked_val_y):
        raise RuntimeError(f"Seed {seed}: VAL labels do not align with J1.")

    if not np.array_equal(yt, locked_test_y):
        raise RuntimeError(f"Seed {seed}: TEST labels do not align with J1.")

    if not np.array_equal(ids_v.astype(str), locked_val_ids):
        raise RuntimeError(f"Seed {seed}: VAL IDs do not align with J1 row order.")

    if not np.array_equal(ids_t.astype(str), locked_test_ids):
        raise RuntimeError(f"Seed {seed}: TEST IDs do not align with J1 row order.")

    val_probs_by_seed[seed] = pv
    test_probs_by_seed[seed] = pt

    val_features_by_seed[seed] = fv
    test_features_by_seed[seed] = ft

    val_ids_by_seed[seed] = ids_v
    test_ids_by_seed[seed] = ids_t

    vm = metrics_dict(yv, pv)
    tm = metrics_dict(yt, pt)

    seed_summaries.append({
        "seed": seed,
        "selected_epoch": int(finetune_best["epoch"]),
        "selected_val_bacc": float(finetune_best["val_bacc"]),
        "selected_val_macro_f1": float(finetune_best["val_macro_f1"]),
        "val_accuracy": vm["accuracy"],
        "val_bacc": vm["bacc"],
        "val_macro_f1": vm["macro_f1"],
        "val_auc": vm["auc"],
        "test_accuracy": tm["accuracy"],
        "test_bacc": tm["bacc"],
        "test_macro_f1": tm["macro_f1"],
        "test_auc": tm["auc"],
    })

    print("\nFrozen seed result:")
    print("VAL :", vm)
    print("TEST:", tm)

    del model, train_loader, optimizer, warmup_optimizer
    gc.collect()
    torch.cuda.empty_cache()

SEED_RESULTS_DF = pd.DataFrame(seed_summaries)
HISTORY_DF = pd.concat(all_histories, ignore_index=True)

display(SEED_RESULTS_DF)

SEED_RESULTS_DF.to_csv(
    TABLE_DIR / "dinov2_multiseed_seed_results.csv",
    index=False,
)
HISTORY_DF.to_csv(
    TABLE_DIR / "dinov2_multiseed_training_history.csv",
    index=False,
)


DINOv2 SEED 21


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
/tmp/ipykernel_23/258848220.py:50: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return Image.fromarray(hsv, mode="HSV").convert("RGB")
/tmp/ipykernel_23/258848220.py:50: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return Image.fromarray(hsv, mode="HSV").convert("RGB")


seed=21 | linear_warmup | epoch 01/3 | loss=1.7409 | VAL BACC=0.4933 | F1=0.4006
seed=21 | linear_warmup | epoch 02/3 | loss=1.4649 | VAL BACC=0.5129 | F1=0.4210
seed=21 | linear_warmup | epoch 03/3 | loss=1.3356 | VAL BACC=0.5682 | F1=0.4480
seed=21 | full_finetune | epoch 01/10 | loss=1.4289 | VAL BACC=0.5736 | F1=0.5459
seed=21 | full_finetune | epoch 02/10 | loss=1.1239 | VAL BACC=0.6045 | F1=0.5633
seed=21 | full_finetune | epoch 03/10 | loss=1.0068 | VAL BACC=0.6339 | F1=0.5551
seed=21 | full_finetune | epoch 04/10 | loss=0.9028 | VAL BACC=0.6547 | F1=0.6175
seed=21 | full_finetune | epoch 05/10 | loss=0.7499 | VAL BACC=0.7158 | F1=0.6463
seed=21 | full_finetune | epoch 06/10 | loss=0.6525 | VAL BACC=0.7177 | F1=0.6908
seed=21 | full_finetune | epoch 07/10 | loss=0.5363 | VAL BACC=0.7434 | F1=0.6930
seed=21 | full_finetune | epoch 08/10 | loss=0.4287 | VAL BACC=0.7743 | F1=0.7213
seed=21 | full_finetune | epoch 09/10 | loss=0.3624 | VAL BACC=0.7765 | F1=0.7431
seed=21 | full_fine

Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
/tmp/ipykernel_23/258848220.py:50: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return Image.fromarray(hsv, mode="HSV").convert("RGB")
/tmp/ipykernel_23/258848220.py:50: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return Image.fromarray(hsv, mode="HSV").convert("RGB")


seed=42 | linear_warmup | epoch 01/3 | loss=1.7462 | VAL BACC=0.5259 | F1=0.4362
seed=42 | linear_warmup | epoch 02/3 | loss=1.4568 | VAL BACC=0.5384 | F1=0.4905
seed=42 | linear_warmup | epoch 03/3 | loss=1.3544 | VAL BACC=0.5565 | F1=0.4330
seed=42 | full_finetune | epoch 01/10 | loss=1.5296 | VAL BACC=0.6077 | F1=0.4492
seed=42 | full_finetune | epoch 02/10 | loss=1.1487 | VAL BACC=0.6233 | F1=0.5163
seed=42 | full_finetune | epoch 03/10 | loss=1.0302 | VAL BACC=0.6391 | F1=0.5604
seed=42 | full_finetune | epoch 04/10 | loss=0.8946 | VAL BACC=0.6784 | F1=0.5970
seed=42 | full_finetune | epoch 05/10 | loss=0.7791 | VAL BACC=0.6861 | F1=0.6308
seed=42 | full_finetune | epoch 06/10 | loss=0.6543 | VAL BACC=0.7077 | F1=0.6936
seed=42 | full_finetune | epoch 07/10 | loss=0.5495 | VAL BACC=0.7854 | F1=0.6816
seed=42 | full_finetune | epoch 08/10 | loss=0.4363 | VAL BACC=0.7748 | F1=0.6972
seed=42 | full_finetune | epoch 09/10 | loss=0.3605 | VAL BACC=0.7683 | F1=0.7216
seed=42 | full_fine

Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
/tmp/ipykernel_23/258848220.py:50: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return Image.fromarray(hsv, mode="HSV").convert("RGB")
/tmp/ipykernel_23/258848220.py:50: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return Image.fromarray(hsv, mode="HSV").convert("RGB")


seed=62 | linear_warmup | epoch 01/3 | loss=1.7336 | VAL BACC=0.4549 | F1=0.4078
seed=62 | linear_warmup | epoch 02/3 | loss=1.4684 | VAL BACC=0.5402 | F1=0.4527
seed=62 | linear_warmup | epoch 03/3 | loss=1.3463 | VAL BACC=0.5715 | F1=0.4552
seed=62 | full_finetune | epoch 01/10 | loss=1.4385 | VAL BACC=0.5780 | F1=0.5141
seed=62 | full_finetune | epoch 02/10 | loss=1.1415 | VAL BACC=0.6327 | F1=0.5749
seed=62 | full_finetune | epoch 03/10 | loss=0.9971 | VAL BACC=0.6293 | F1=0.5044
seed=62 | full_finetune | epoch 04/10 | loss=0.8844 | VAL BACC=0.6916 | F1=0.6156
seed=62 | full_finetune | epoch 05/10 | loss=0.7751 | VAL BACC=0.6958 | F1=0.6467
seed=62 | full_finetune | epoch 06/10 | loss=0.6545 | VAL BACC=0.7314 | F1=0.6736
seed=62 | full_finetune | epoch 07/10 | loss=0.5353 | VAL BACC=0.7468 | F1=0.6878
seed=62 | full_finetune | epoch 08/10 | loss=0.4208 | VAL BACC=0.7619 | F1=0.7353
seed=62 | full_finetune | epoch 09/10 | loss=0.3466 | VAL BACC=0.7716 | F1=0.7353
seed=62 | full_fine

   seed  selected_epoch  selected_val_bacc  selected_val_macro_f1  \
0    21              10           0.778775               0.737670   
1    42              10           0.788204               0.734830   
2    62               9           0.771643               0.735255   

   val_accuracy  val_bacc  val_macro_f1   val_auc  test_accuracy  test_bacc  \
0      0.788393  0.778775      0.737670  0.968661       0.791239   0.798667   
1      0.790762  0.788204      0.734830  0.970878       0.780979   0.788626   
2      0.779313  0.771643      0.735255  0.969770       0.771507   0.797952   

   test_macro_f1  test_auc  
0       0.772480  0.970567  
1       0.766607  0.969631  
2       0.769293  0.969001  

In [12]:
# ============================================================
# CELL 11 — MULTI-SEED MEAN PROBABILITY BASELINE
# ============================================================

Y_VAL = VAL_DF["label"].to_numpy(np.int32)
Y_TEST = TEST_DF["label"].to_numpy(np.int32)

DINO_VAL_PROB = normalize_prob(
    np.mean(
        np.stack([val_probs_by_seed[s] for s in SEEDS], axis=0),
        axis=0,
    )
)

DINO_TEST_PROB = normalize_prob(
    np.mean(
        np.stack([test_probs_by_seed[s] for s in SEEDS], axis=0),
        axis=0,
    )
)

DINO_VAL_METRICS = metrics_dict(
    Y_VAL,
    DINO_VAL_PROB,
)

DINO_TEST_METRICS = metrics_dict(
    Y_TEST,
    DINO_TEST_PROB,
)

DINO_MEAN_DF = pd.DataFrame([
    {
        "model": "DINOv2 ViT-S/14 3-seed mean",
        "split": "Validation",
        "n": len(Y_VAL),
        **DINO_VAL_METRICS,
    },
    {
        "model": "DINOv2 ViT-S/14 3-seed mean",
        "split": "InternalTest",
        "n": len(Y_TEST),
        **DINO_TEST_METRICS,
    },
])

display(DINO_MEAN_DF)

DINO_MEAN_DF.to_csv(
    TABLE_DIR / "dinov2_multiseed_mean_results.csv",
    index=False,
)

np.savez_compressed(
    OUTPUT_DIR / "dinov2_original_split_multiseed_probabilities.npz",
    y_val=Y_VAL,
    y_test=Y_TEST,
    val_image_id=VAL_DF["image_id"].astype(str).to_numpy(),
    test_image_id=TEST_DF["image_id"].astype(str).to_numpy(),
    val_prob=DINO_VAL_PROB,
    test_prob=DINO_TEST_PROB,
    seeds=np.asarray(SEEDS, dtype=np.int32),
)

print(
    "DINOv2 3-seed mean TEST BACC:",
    f"{100 * DINO_TEST_METRICS['bacc']:.4f}%"
)

                         model         split     n  accuracy      bacc  \
0  DINOv2 ViT-S/14 3-seed mean    Validation  2533  0.798658  0.791380   
1  DINOv2 ViT-S/14 3-seed mean  InternalTest  2534  0.796369  0.804301   

   macro_precision  macro_recall  macro_f1  weighted_f1       auc  
0         0.713614      0.791380  0.746746     0.803427  0.973882  
1         0.778299      0.804301  0.788338     0.801076  0.974008  

DINOv2 3-seed mean TEST BACC: 80.4301%


In [13]:
# ============================================================
# CELL 12 — SELECT REPRESENTATIVE DINO SEED BY VALIDATION ONLY
# FOR LATER CKA / UMAP ANALYSIS
# ============================================================

selection_df = SEED_RESULTS_DF.sort_values(
    [
        "val_bacc",
        "val_macro_f1",
        "val_accuracy",
        "seed",
    ],
    ascending=[False, False, False, True],
).reset_index(drop=True)

REPRESENTATIVE_SEED = int(
    selection_df.iloc[0]["seed"]
)

print("Representative seed selected on validation only:", REPRESENTATIVE_SEED)

REP_VAL_FEATURES = val_features_by_seed[REPRESENTATIVE_SEED]
REP_TEST_FEATURES = test_features_by_seed[REPRESENTATIVE_SEED]

np.savez_compressed(
    FEATURE_DIR / "dinov2_representative_val_features.npz",
    seed=np.asarray([REPRESENTATIVE_SEED]),
    image_id=VAL_DF["image_id"].astype(str).to_numpy(),
    labels=Y_VAL,
    features=REP_VAL_FEATURES,
    probabilities=val_probs_by_seed[REPRESENTATIVE_SEED],
)

np.savez_compressed(
    FEATURE_DIR / "dinov2_representative_test_features.npz",
    seed=np.asarray([REPRESENTATIVE_SEED]),
    image_id=TEST_DF["image_id"].astype(str).to_numpy(),
    labels=Y_TEST,
    features=REP_TEST_FEATURES,
    probabilities=test_probs_by_seed[REPRESENTATIVE_SEED],
)

selection_df.to_csv(
    TABLE_DIR / "dinov2_representative_seed_validation_selection.csv",
    index=False,
)

print("Representative feature shape:")
print("VAL :", REP_VAL_FEATURES.shape)
print("TEST:", REP_TEST_FEATURES.shape)

Representative seed selected on validation only: 42
Representative feature shape:
VAL : (2533, 384)
TEST: (2534, 384)


In [14]:
# ============================================================
# CELL 13 — DIRECT COMPARISON WITH FROZEN 83.95% PROPOSED MODEL
# ============================================================

with np.load(FINAL_PROB_NPZ) as z:
    proposed_y_test = np.asarray(
        z["y_test"],
        dtype=np.int32,
    )
    proposed_test_prob = normalize_prob(
        z["final_test_prob"]
    )
    proposed_y_val = np.asarray(
        z["y_val"],
        dtype=np.int32,
    )
    proposed_val_prob = normalize_prob(
        z["final_val_prob"]
    )

if not np.array_equal(proposed_y_test, Y_TEST):
    raise RuntimeError(
        "Frozen proposed-model test labels do not align with J1/J2."
    )

if not np.array_equal(proposed_y_val, Y_VAL):
    raise RuntimeError(
        "Frozen proposed-model validation labels do not align with J1/J2."
    )

PROPOSED_TEST_METRICS = metrics_dict(
    Y_TEST,
    proposed_test_prob,
)

PROPOSED_VAL_METRICS = metrics_dict(
    Y_VAL,
    proposed_val_prob,
)

comparison_rows = [
    {
        "model": "Final proposed ensemble",
        "selection": "Original validation protocol",
        **PROPOSED_TEST_METRICS,
    },
    {
        "model": "DINOv2 ViT-S/14 3-seed mean",
        "selection": "Original validation protocol",
        **DINO_TEST_METRICS,
    },
]

# Add original comparison-table rows when available.
if COMPARISON_CSV is not None:
    original_comparison = pd.read_csv(COMPARISON_CSV)
    original_comparison.to_csv(
        TABLE_DIR / "original_thesis_internal_comparison_copy.csv",
        index=False,
    )

COMPARISON_DF = pd.DataFrame(comparison_rows)

display(COMPARISON_DF.sort_values("bacc", ascending=False))

COMPARISON_DF.to_csv(
    TABLE_DIR / "dinov2_vs_frozen_proposed_same_test.csv",
    index=False,
)

print("\nBACC difference (Proposed - DINO):")
print(
    f"{100 * (PROPOSED_TEST_METRICS['bacc'] - DINO_TEST_METRICS['bacc']):+.4f} percentage points"
)

                         model                     selection  accuracy  \
0      Final proposed ensemble  Original validation protocol  0.838990   
1  DINOv2 ViT-S/14 3-seed mean  Original validation protocol  0.796369   

       bacc  macro_precision  macro_recall  macro_f1  weighted_f1       auc  
0  0.839549         0.766652      0.839549  0.797961     0.841509  0.976656  
1  0.804301         0.778299      0.804301  0.788338     0.801076  0.974008  


BACC difference (Proposed - DINO):
+3.5248 percentage points


In [15]:
# ============================================================
# CELL 14 — CLASSIFICATION REPORT + CONFUSION MATRIX
# ============================================================

DINO_TEST_PRED = np.argmax(
    DINO_TEST_PROB,
    axis=1,
)

report = classification_report(
    Y_TEST,
    DINO_TEST_PRED,
    labels=np.arange(NUM_CLASSES),
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)

REPORT_DF = pd.DataFrame(report).T
display(REPORT_DF)

REPORT_DF.to_csv(
    TABLE_DIR / "dinov2_multiseed_test_classification_report.csv"
)

cm = confusion_matrix(
    Y_TEST,
    DINO_TEST_PRED,
    labels=np.arange(NUM_CLASSES),
)

plt.figure(figsize=(7.4, 6.3))
plt.imshow(cm, interpolation="nearest")
plt.title("DINOv2 ViT-S/14 — Original Internal Test")
plt.colorbar()

ticks = np.arange(NUM_CLASSES)
plt.xticks(ticks, CLASS_NAMES, rotation=45, ha="right")
plt.yticks(ticks, CLASS_NAMES)
plt.xlabel("Predicted")
plt.ylabel("True")

threshold = cm.max() / 2.0

for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        plt.text(
            j, i, str(cm[i, j]),
            ha="center", va="center",
            color="white" if cm[i, j] > threshold else "black",
            fontsize=8,
        )

plt.tight_layout()
plt.savefig(
    FIG_DIR / "dinov2_original_test_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

              precision    recall  f1-score      support
MEL            0.633962  0.743363  0.684318   452.000000
NV             0.913119  0.815994  0.861829  1288.000000
BCC            0.827485  0.849850  0.838519   333.000000
AK             0.565217  0.755814  0.646766    86.000000
BKL            0.671280  0.737643  0.702899   263.000000
DF             0.869565  0.833333  0.851064    24.000000
VASC           1.000000  1.000000  1.000000    25.000000
SCC            0.745763  0.698413  0.721311    63.000000
accuracy       0.796369  0.796369  0.796369     0.796369
macro avg      0.778299  0.804301  0.788338  2534.000000
weighted avg   0.811448  0.796369  0.801076  2534.000000

In [16]:
# ============================================================
# CELL 15 — TRAINING CURVES
# ============================================================

plt.figure(figsize=(9.0, 5.3))

for seed in SEEDS:
    g = HISTORY_DF.loc[
        (HISTORY_DF["seed"] == seed)
        & (HISTORY_DF["phase"] == "full_finetune")
    ].copy()

    plt.plot(
        g["epoch"],
        100 * g["val_bacc"],
        marker="o",
        label=f"seed {seed}",
    )

plt.xlabel("Fine-tuning epoch")
plt.ylabel("Validation Balanced Accuracy (%)")
plt.title("DINOv2 Validation-Only Checkpoint Selection")
plt.grid(alpha=0.2)
plt.legend()
plt.tight_layout()

plt.savefig(
    FIG_DIR / "dinov2_multiseed_validation_bacc_curves.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [17]:
# ============================================================
# CELL 16 — SELECTION / REPRODUCIBILITY AUDIT
# ============================================================

audit = {
    "experiment": "J2 DINOv2 modern SSL baseline",
    "protocol_source": str(J1_MARKER),
    "protocol_hash": J1_INFO.get("combined_protocol_sha256"),
    "split": "original thesis stratified 80/10/10",
    "split_sizes": {
        "train": len(TRAIN_DF),
        "validation": len(VAL_DF),
        "test": len(TEST_DF),
    },
    "model": DINO_MODEL_NAME,
    "seeds": SEEDS,
    "linear_warmup_epochs_max": LINEAR_WARMUP_EPOCHS,
    "finetune_epochs_max": FINETUNE_EPOCHS,
    "patience": PATIENCE,
    "checkpoint_selection": "validation BACC; macro-F1 tie-break",
    "test_used_for_checkpoint_selection": False,
    "multiseed_combination": "arithmetic mean of per-seed softmax probabilities",
    "representative_seed": REPRESENTATIVE_SEED,
    "representative_seed_selection": "validation only",
    "torch_version": torch.__version__,
    "torch_cuda_version": torch.version.cuda,
    "pillow_version": PIL.__version__,
    "torchvision_used": False,
    "gpu_name": torch.cuda.get_device_name(0),
    "gpu_capability": list(torch.cuda.get_device_capability(0)),
    "batch_size": BATCH_SIZE,
    "dinov2_validation_metrics": DINO_VAL_METRICS,
    "dinov2_test_metrics": DINO_TEST_METRICS,
    "proposed_test_metrics": PROPOSED_TEST_METRICS,
}

with open(
    AUDIT_DIR / "J2_DINO_ORIGINAL_SPLIT_AUDIT.json",
    "w",
) as f:
    json.dump(
        audit,
        f,
        indent=2,
        default=float,
    )

print(json.dumps(audit, indent=2, default=float))

{
  "experiment": "J2 DINOv2 modern SSL baseline",
  "protocol_source": "/kaggle/input/notebooks/anirbanroysudipto/journal-extension-j1/J1_PROTOCOL_LOCK_COMPLETE.json",
  "protocol_hash": "b483ced043583491ed0793a5e80ecd1192055230393b6603200a3441d04f5766",
  "split": "original thesis stratified 80/10/10",
  "split_sizes": {
    "train": 20264,
    "validation": 2533,
    "test": 2534
  },
  "model": "dinov2_vits14",
  "seeds": [
    21,
    42,
    62
  ],
  "linear_warmup_epochs_max": 3,
  "finetune_epochs_max": 10,
  "patience": 3,
  "checkpoint_selection": "validation BACC; macro-F1 tie-break",
  "test_used_for_checkpoint_selection": false,
  "multiseed_combination": "arithmetic mean of per-seed softmax probabilities",
  "representative_seed": 42,
  "representative_seed_selection": "validation only",
  "torch_version": "2.7.1+cu118",
  "torch_cuda_version": "11.8",
  "pillow_version": "11.3.0",
  "torchvision_used": false,
  "gpu_name": "Tesla P100-PCIE-16GB",
  "gpu_capability": [
 

In [18]:
# ============================================================
# CELL 17 — COMPLETION MARKER + ZIP PACKAGE
# ============================================================

marker = {
    "stage": "J2_dinov2_original_split",
    "status": "complete",
    "protocol": "original_stratified_80_10_10",
    "protocol_hash": J1_INFO.get("combined_protocol_sha256"),
    "model": DINO_MODEL_NAME,
    "seeds": SEEDS,
    "representative_seed": REPRESENTATIVE_SEED,
    "test_used_for_selection": False,
    "dinov2_test_bacc": float(DINO_TEST_METRICS["bacc"]),
    "proposed_test_bacc": float(PROPOSED_TEST_METRICS["bacc"]),
    "output_rel": "j2_dinov2_original_split",
    "probabilities_rel": (
        "j2_dinov2_original_split/"
        "dinov2_original_split_multiseed_probabilities.npz"
    ),
    "val_features_rel": (
        "j2_dinov2_original_split/features/"
        "dinov2_representative_val_features.npz"
    ),
    "test_features_rel": (
        "j2_dinov2_original_split/features/"
        "dinov2_representative_test_features.npz"
    ),
}

MARKER_PATH = WORKING / "J2_DINO_ORIGINAL_SPLIT_COMPLETE.json"

with open(MARKER_PATH, "w") as f:
    json.dump(marker, f, indent=2)

shutil.copy2(
    MARKER_PATH,
    OUTPUT_DIR / "J2_DINO_ORIGINAL_SPLIT_COMPLETE.json",
)

ZIP_PATH = WORKING / "J2_DINO_ORIGINAL_SPLIT_RESULTS.zip"

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    for p in sorted(OUTPUT_DIR.rglob("*")):
        if p.is_file():
            # Large model checkpoints are intentionally excluded
            # from the portable results ZIP.
            if p.parent == MODEL_DIR:
                continue

            zf.write(
                p,
                arcname=str(p.relative_to(OUTPUT_DIR.parent)),
            )

required = [
    OUTPUT_DIR / "dinov2_original_split_multiseed_probabilities.npz",
    TABLE_DIR / "dinov2_multiseed_seed_results.csv",
    TABLE_DIR / "dinov2_multiseed_mean_results.csv",
    TABLE_DIR / "dinov2_vs_frozen_proposed_same_test.csv",
    FEATURE_DIR / "dinov2_representative_val_features.npz",
    FEATURE_DIR / "dinov2_representative_test_features.npz",
    AUDIT_DIR / "J2_DINO_ORIGINAL_SPLIT_AUDIT.json",
    MARKER_PATH,
    ZIP_PATH,
]

missing = [str(p) for p in required if not p.exists()]

if missing:
    raise FileNotFoundError(
        "Missing J2 outputs:\n" + "\n".join(missing)
    )

print("=" * 96)
print("J2 DINOv2 ORIGINAL-SPLIT BASELINE COMPLETE")
print("=" * 96)
print("Protocol hash :", J1_INFO.get("combined_protocol_sha256"))
print("DINO seeds    :", SEEDS)
print("Representative:", REPRESENTATIVE_SEED)
print("DINO TEST BACC:", f"{100 * DINO_TEST_METRICS['bacc']:.4f}%")
print("PROP TEST BACC:", f"{100 * PROPOSED_TEST_METRICS['bacc']:.4f}%")
print(
    "Difference    :",
    f"{100 * (PROPOSED_TEST_METRICS['bacc'] - DINO_TEST_METRICS['bacc']):+.4f} pp"
)
print("ZIP package   :", ZIP_PATH)
print()
print("Save this Kaggle notebook Version WITH OUTPUTS.")
print("Then give the executed notebook/output back for J3 ablation.")

J2 DINOv2 ORIGINAL-SPLIT BASELINE COMPLETE
Protocol hash : b483ced043583491ed0793a5e80ecd1192055230393b6603200a3441d04f5766
DINO seeds    : [21, 42, 62]
Representative: 42
DINO TEST BACC: 80.4301%
PROP TEST BACC: 83.9549%
Difference    : +3.5248 pp
ZIP package   : /kaggle/working/J2_DINO_ORIGINAL_SPLIT_RESULTS.zip

Save this Kaggle notebook Version WITH OUTPUTS.
Then give the executed notebook/output back for J3 ablation.
